In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [3]:
Path("../outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("../outputs/models").mkdir(parents=True, exist_ok=True)

In [4]:
with open("../data/graph/train_graph.pkl", "rb") as f:
    train_graph = pickle.load(f)

with open("../data/graph/test_graph.pkl", "rb") as f:
    test_graph = pickle.load(f)

with open("../data/graph/wallet_mapping.pkl", "rb") as f:
    wallet_mapping = pickle.load(f)

num_nodes = wallet_mapping["num_nodes"]

print("Num nodes:", num_nodes)
print("Train edges:", train_graph["edge_index"].shape)
print("Test edges:", test_graph["edge_index"].shape)

Num nodes: 108582
Train edges: (2, 38454)
Test edges: (2, 298160)


In [5]:
class EdgeDataset(Dataset):
    def __init__(self, graph):
        self.edge_index = torch.tensor(graph["edge_index"], dtype=torch.long)
        self.edge_features = torch.tensor(graph["edge_features"], dtype=torch.float32)
        self.labels = torch.tensor(graph["edge_labels"], dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        source = self.edge_index[0, idx]
        target = self.edge_index[1, idx]
        edge_feat = self.edge_features[idx]
        label = self.labels[idx]

        return source, target, edge_feat, label

In [6]:
BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_graph)
test_dataset = EdgeDataset(test_graph)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [7]:
class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)

        input_dim = embedding_dim * 2 + edge_feat_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        tgt_emb = self.node_embedding(target)

        x = torch.cat([src_emb, tgt_emb, edge_feat], dim=1)

        logits = self.mlp(x).squeeze(1)

        return logits

In [8]:
edge_feat_dim = train_graph["edge_features"].shape[1]

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(108582, 64)
  (mlp): Sequential(
    (0): Linear(in_features=138, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [9]:
labels = train_graph["edge_labels"]

num_positive = np.sum(labels == 1)
num_negative = np.sum(labels == 0)

pos_weight_value = num_negative / num_positive

print("Positive:", num_positive)
print("Negative:", num_negative)
print("pos_weight:", pos_weight_value)

Positive: 754
Negative: 37700
pos_weight: 50.0


In [10]:
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [11]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [12]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [22]:
def evaluate(model, loader, threshold=0.5):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(
                source,
                target,
                edge_feat
            )

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                label.numpy()
            )

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, preds)
    }

    return metrics

In [14]:
EPOCHS = 20

history = []

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_metrics = evaluate(model, test_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | "
        f"Rec: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f} | "
        f"ROC-AUC: {test_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {test_metrics['pr_auc']:.4f}"
    )

    if test_metrics["f1"] > best_f1:
        best_f1 = test_metrics["f1"]

        torch.save(
            model.state_dict(),
            "../outputs/models/graph_class_weight_best.pt"
        )

Epoch 01 | Loss: 1.2622 | Acc: 0.5699 | Prec: 0.0004 | Rec: 0.7656 | F1: 0.0008 | ROC-AUC: 0.7364 | PR-AUC: 0.0023
Epoch 02 | Loss: 1.0773 | Acc: 0.7427 | Prec: 0.0005 | Rec: 0.6250 | F1: 0.0010 | ROC-AUC: 0.7642 | PR-AUC: 0.0018
Epoch 03 | Loss: 0.9391 | Acc: 0.7396 | Prec: 0.0005 | Rec: 0.6562 | F1: 0.0011 | ROC-AUC: 0.7736 | PR-AUC: 0.0016
Epoch 04 | Loss: 0.8478 | Acc: 0.7861 | Prec: 0.0005 | Rec: 0.5469 | F1: 0.0011 | ROC-AUC: 0.7611 | PR-AUC: 0.0018
Epoch 05 | Loss: 0.7189 | Acc: 0.7953 | Prec: 0.0006 | Rec: 0.5312 | F1: 0.0011 | ROC-AUC: 0.7625 | PR-AUC: 0.0026
Epoch 06 | Loss: 0.5992 | Acc: 0.8224 | Prec: 0.0006 | Rec: 0.5156 | F1: 0.0012 | ROC-AUC: 0.7650 | PR-AUC: 0.0072
Epoch 07 | Loss: 0.4712 | Acc: 0.8614 | Prec: 0.0006 | Rec: 0.3594 | F1: 0.0011 | ROC-AUC: 0.7451 | PR-AUC: 0.0495
Epoch 08 | Loss: 0.3905 | Acc: 0.8877 | Prec: 0.0006 | Rec: 0.3281 | F1: 0.0013 | ROC-AUC: 0.7431 | PR-AUC: 0.0491
Epoch 09 | Loss: 0.2940 | Acc: 0.8990 | Prec: 0.0007 | Rec: 0.3438 | F1: 0.0015 

In [15]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../outputs/metrics/graph_class_weight_history.csv",
    index=False
)

history_df

,epoch,train_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,1.262245,0.569858,0.000382,0.765625,0.000764,0.736440,0.002267
1,2,1.077291,0.742682,0.000521,0.625000,0.001042,0.764241,0.001772
2,3,0.939136,0.739603,0.000541,0.656250,0.001081,0.773564,0.001614
3,4,0.847812,0.786138,0.000549,0.546875,0.001097,0.761056,0.001782
4,5,0.718947,0.795328,0.000557,0.531250,0.001113,0.762477,0.002595
5,6,0.599198,0.822401,0.000623,0.515625,0.001245,0.764984,0.007153
6,7,0.471230,0.861407,0.000557,0.359375,0.001112,0.745075,0.049521
7,8,0.390462,0.887741,0.000628,0.328125,0.001253,0.743056,0.049138
8,9,0.293965,0.899007,0.000731,0.343750,0.001459,0.760891,0.070374
9,10,0.238251,0.919617,0.000752,0.281250,0.001500,0.740210,0.070561


In [16]:
model.load_state_dict(
    torch.load("../outputs/models/graph_class_weight_best.pt")
)

final_metrics = evaluate(model, test_loader)

print("Final Metrics:")
for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(k, ":", v)

print("Confusion Matrix:")
print(final_metrics["confusion_matrix"])

Final Metrics:
threshold : 0.5
accuracy : 0.9705963241212772
precision : 0.0018317115054378936
recall : 0.25
f1 : 0.003636776906466644
roc_auc : 0.7581635299534378
pr_auc : 0.06566412969505993
Confusion Matrix:
[[289377   8719]
 [    48     16]]


In [17]:
threshold_results = []

thresholds = [0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.2, 0.3, 0.4, 0.5]

for t in thresholds:
    m = evaluate(model, test_loader, threshold=t)
    cm = m["confusion_matrix"]

    threshold_results.append({
        "threshold": t,
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "roc_auc": m["roc_auc"],
        "pr_auc": m["pr_auc"],
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.01,0.735162,0.000506,0.625000,0.001012,0.758164,0.065664,219156,78940,24,40
1,0.02,0.800453,0.000588,0.546875,0.001175,0.758164,0.065664,238628,59468,29,35
2,0.03,0.832794,0.000642,0.500000,0.001282,0.758164,0.065664,248274,49822,32,32
3,0.05,0.867964,0.000711,0.437500,0.001420,0.758164,0.065664,258764,39332,36,28
4,0.07,0.887601,0.000806,0.421875,0.001609,0.758164,0.065664,264620,33476,37,27
5,0.10,0.906282,0.000931,0.406250,0.001857,0.758164,0.065664,270191,27905,38,26
6,0.20,0.936890,0.001064,0.312500,0.002121,0.758164,0.065664,279323,18773,44,20
7,0.30,0.952150,0.001194,0.265625,0.002377,0.758164,0.065664,283876,14220,47,17
8,0.40,0.962587,0.001438,0.250000,0.002860,0.758164,0.065664,286989,11107,48,16
9,0.50,0.970596,0.001832,0.250000,0.003637,0.758164,0.065664,289377,8719,48,16


In [18]:
threshold_df.sort_values(by="recall", ascending=False)

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.01,0.735162,0.000506,0.625000,0.001012,0.758164,0.065664,219156,78940,24,40
1,0.02,0.800453,0.000588,0.546875,0.001175,0.758164,0.065664,238628,59468,29,35
2,0.03,0.832794,0.000642,0.500000,0.001282,0.758164,0.065664,248274,49822,32,32
3,0.05,0.867964,0.000711,0.437500,0.001420,0.758164,0.065664,258764,39332,36,28
4,0.07,0.887601,0.000806,0.421875,0.001609,0.758164,0.065664,264620,33476,37,27
5,0.10,0.906282,0.000931,0.406250,0.001857,0.758164,0.065664,270191,27905,38,26
6,0.20,0.936890,0.001064,0.312500,0.002121,0.758164,0.065664,279323,18773,44,20
7,0.30,0.952150,0.001194,0.265625,0.002377,0.758164,0.065664,283876,14220,47,17
8,0.40,0.962587,0.001438,0.250000,0.002860,0.758164,0.065664,286989,11107,48,16
9,0.50,0.970596,0.001832,0.250000,0.003637,0.758164,0.065664,289377,8719,48,16


In [24]:
# =========================
# THRESHOLD TUNING ANALYSIS
# =========================

BEST_THRESHOLD = 0.01

# Evaluation mode
model.eval()

# Container
all_labels = []
all_probs = []

# Inference
with torch.no_grad():

    for source, target, edge_feat, label in test_loader:

        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)

        logits = model(
            source,
            target,
            edge_feat
        )

        probs = torch.sigmoid(logits)

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            label.numpy()
        )

# Convert to numpy
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Create dataframe
test_results = pd.DataFrame({
    "probability": all_probs,
    "true_label": all_labels
})

# Apply threshold
test_results["pred_label"] = (
    test_results["probability"] >= BEST_THRESHOLD
).astype(int)

# Candidate fraud
candidate_fraud = test_results[
    test_results["pred_label"] == 1
]

# Evaluation
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

cm = confusion_matrix(
    test_results["true_label"],
    test_results["pred_label"]
)

print("=" * 50)
print(f"THRESHOLD : {BEST_THRESHOLD}")
print("=" * 50)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    test_results["true_label"],
    test_results["pred_label"],
    zero_division=0
))

print("\nCandidate Fraud Shape:")
print(candidate_fraud.shape)

print("\nCandidate Fraud Preview:")
display(candidate_fraud.head())

# Save result
test_results.to_csv(
    "../outputs/metrics/graph_class_weight_predictions_threshold_001.csv",
    index=False
)

print("\nSaved:")
print("../outputs/metrics/graph_class_weight_predictions_threshold_001.csv")

THRESHOLD : 0.01

Confusion Matrix:
[[219156  78940]
 [    24     40]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.74      0.85    298096
         1.0       0.00      0.62      0.00        64

    accuracy                           0.74    298160
   macro avg       0.50      0.68      0.42    298160
weighted avg       1.00      0.74      0.85    298160


Candidate Fraud Shape:
(78980, 3)

Candidate Fraud Preview:


,probability,true_label,pred_label
11,0.211778,0.0,1
16,0.054236,0.0,1
21,0.309095,0.0,1
23,0.045339,0.0,1
25,0.011045,0.0,1



Saved:
../outputs/metrics/graph_class_weight_predictions_threshold_001.csv


In [26]:
# =========================
# STAGE 2: GABUNG PREDICTION DENGAN DATA TEST ASLI
# =========================

# 1. Load ulang test data asli
test_df = pd.read_csv(
    "../data/processed/test_temporal.csv",
    low_memory=False
)

# 2. Urutkan sama seperti saat build graph dataset
test_df = test_df.sort_values("timestamp").reset_index(drop=True)

# 3. Reset index prediction
test_results = test_results.reset_index(drop=True)

# 4. Cek apakah jumlah baris sama
print("test_df shape      :", test_df.shape)
print("test_results shape :", test_results.shape)

assert len(test_df) == len(test_results), "Jumlah baris test_df dan test_results tidak sama!"

# 5. Gabungkan data asli + hasil prediksi model
test_with_pred = pd.concat(
    [test_df, test_results],
    axis=1
)

# 6. Ambil kandidat fraud dari model threshold 0.01
candidate_fraud_full = test_with_pred[
    test_with_pred["pred_label"] == 1
].copy()

print("\nCandidate fraud full shape:")
print(candidate_fraud_full.shape)

print("\nColumns:")
print(candidate_fraud_full.columns.tolist())

display(candidate_fraud_full.head())

test_df shape      : (298160, 25)
test_results shape : (298160, 3)

Candidate fraud full shape:
(78980, 28)

Columns:
['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt', 'month', 'is_wash_trading', 'rule_self_trade', 'rule_seller_buyback', 'rule_multi_hop_cycle', 'rule_high_pair_count', 'wash_score', 'confidence_category', 'label_final', 'probability', 'true_label', 'pred_label']


,transaction_hash,block_number,timestamp,nft_address,token_id,from_address,to_address,transaction_value,mint_timestamp,transfers_out_from,...,rule_self_trade,rule_seller_buyback,rule_multi_hop_cycle,rule_high_pair_count,wash_score,confidence_category,label_final,probability,true_label,pred_label
11,0xea6f9a4bfb019428be602e572a3f5378cade0c1f64e7...,13100750,1629978077,0xF8C18Df7509c03b45e6247b2b9E73fcaDEF24dd6,7849,0x95F6915a3839c284bAC794a85032555aA0B56226,0xc6B4a892433A682Eb97DBd6CeBd9372b0f0fE05f,1.199000e+17,1.629913e+09,3,...,0,0,0,0,0,normal,0,0.211778,0.0,1
16,0x9c493d6e302f9f690c40cb3d87fe1f5a6d2678317060...,13100750,1629978077,0xa7d8d9ef8D8Ce8992Df33D8b8CF4Aebabd5bD270,122000116,0x0539b872dF4E0bb8a7ff8566d61Aa08530009e0b,0x0e1c3d5357c34F5CbA26D9Bf6f750cBA14Bb2653,4.480000e+17,1.628014e+09,30,...,0,0,0,0,0,normal,0,0.054236,0.0,1
21,0x3342bbcd17ef2ad4b48ce40a28d64073d997addc2cdd...,13100751,1629978108,0x46F9A4522666d2476a5F5Cd51ea3E0b5800E7f98,739,0xfcc5bA7C5a2dC4eF495F64064BC6C9491bb78bcD,0xF5D7efc534CaA148D8421CebbF1d308acF140aD9,1.700000e+17,NaN,93,...,0,0,0,0,0,normal,0,0.309095,0.0,1
23,0x60ef99aa6b9895adb792d704b0a495c3c00c41b58460...,13100751,1629978108,0x2BD60F290060451e3644a7559D520C2e9b32C7e9,3609,0x2dc55C094EE20b08EA028e56c9dD628AA050Ceb8,0xEE43891FeF06E94d8695Ec128B7aeD956c656a1e,5.400000e+16,1.629226e+09,113,...,0,0,0,0,0,normal,0,0.045339,0.0,1
25,0x31b64e48f9bd414528b85ea0baf8cd01925e340f4aee...,13100752,1629978121,0x343f999eAACdFa1f201fb8e43ebb35c99D9aE0c1,3291,0x011ea68c15f4a8316Da45C1E7844311CdD0ba149,0x9dF05Adf5119d34d16562A7582cf7121350D931e,7.000000e+17,1.626623e+09,1331,...,0,0,0,0,0,normal,0,0.011045,0.0,1


In [27]:
# =========================
# STAGE 2: RULE-BASED FILTERING
# =========================

from sklearn.metrics import confusion_matrix, classification_report

filter_results = []

rules = [
    ("model_only", candidate_fraud_full),
    ("wash_score >= 1", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 1]),
    ("wash_score >= 2", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 2]),
    ("wash_score >= 3", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 3]),
    ("confidence medium/high", candidate_fraud_full[
        candidate_fraud_full["confidence_category"].isin(["medium", "high"])
    ]),
    ("confidence high", candidate_fraud_full[
        candidate_fraud_full["confidence_category"] == "high"
    ]),
]

total_real_fraud = test_with_pred["true_label"].sum()

for rule_name, filtered_df in rules:
    tp = filtered_df[filtered_df["true_label"] == 1].shape[0]
    fp = filtered_df[filtered_df["true_label"] == 0].shape[0]
    fn = total_real_fraud - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    filter_results.append({
        "filter": rule_name,
        "candidate_count": len(filtered_df),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

filter_df = pd.DataFrame(filter_results)

display(
    filter_df.sort_values(
        by=["recall", "precision"],
        ascending=False
    )
)

# Simpan hasil stage 2
filter_df.to_csv(
    "../outputs/metrics/stage2_rule_filtering_results.csv",
    index=False
)

,filter,candidate_count,tp,fp,fn,precision,recall,f1
0,model_only,78980,40,78940,24.0,0.000506,0.625,0.001012
1,wash_score >= 1,16,0,16,64.0,0.000000,0.000,0.000000
2,wash_score >= 2,16,0,16,64.0,0.000000,0.000,0.000000
3,wash_score >= 3,0,0,0,64.0,0.000000,0.000,0.000000
4,confidence medium/high,0,0,0,64.0,0.000000,0.000,0.000000
5,confidence high,0,0,0,64.0,0.000000,0.000,0.000000


In [28]:
# =========================
# STAGE 2 ALTERNATIF:
# TOP-K RANKING BERDASARKAN PROBABILITY
# =========================

topk_results = []

k_values = [
    100,
    500,
    1000,
    5000,
    10000,
    20000,
    50000,
    len(candidate_fraud_full)
]

total_real_fraud = test_with_pred["true_label"].sum()

for k in k_values:
    topk_df = candidate_fraud_full.sort_values(
        by="probability",
        ascending=False
    ).head(k)

    tp = topk_df[topk_df["true_label"] == 1].shape[0]
    fp = topk_df[topk_df["true_label"] == 0].shape[0]
    fn = total_real_fraud - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    topk_results.append({
        "top_k": k,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

topk_df = pd.DataFrame(topk_results)

display(topk_df)

topk_df.to_csv(
    "../outputs/metrics/stage2_topk_probability_results.csv",
    index=False
)

,top_k,tp,fp,fn,precision,recall,f1
0,100,5,95,59.0,0.050000,0.078125,0.060976
1,500,7,493,57.0,0.014000,0.109375,0.024823
2,1000,7,993,57.0,0.007000,0.109375,0.013158
3,5000,12,4988,52.0,0.002400,0.187500,0.004739
4,10000,16,9984,48.0,0.001600,0.250000,0.003180
5,20000,25,19975,39.0,0.001250,0.390625,0.002492
6,50000,32,49968,32.0,0.000640,0.500000,0.001278
7,78980,40,78940,24.0,0.000506,0.625000,0.001012


In [19]:
final_result = {
    "model": "graph_class_weight",
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": final_metrics["confusion_matrix"][0, 0],
    "fp": final_metrics["confusion_matrix"][0, 1],
    "fn": final_metrics["confusion_matrix"][1, 0],
    "tp": final_metrics["confusion_matrix"][1, 1],
}

pd.DataFrame([final_result]).to_csv(
    "../outputs/metrics/graph_class_weight_final.csv",
    index=False
)

final_result

{'model': 'graph_class_weight',
 'accuracy': 0.9705963241212772,
 'precision': 0.0018317115054378936,
 'recall': 0.25,
 'f1': 0.003636776906466644,
 'roc_auc': 0.7581635299534378,
 'pr_auc': 0.06566412969505993,
 'tn': np.int64(289377),
 'fp': np.int64(8719),
 'fn': np.int64(48),
 'tp': np.int64(16)}